# Regression 

## Setup

### Imports

In [1]:
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

### Load Datasets

In [2]:
base_dir = os.path.join(os.getcwd(), '..')
processed_data_dir = os.path.join(base_dir, 'processed_data')

open_meteo_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')
open_meteo_df = pd.read_csv(open_meteo_filepath, parse_dates=['datetime'])

ausgrid_filepath = os.path.join(processed_data_dir, 'ausgrid_processed_data.csv')
ausgrid_df = pd.read_csv(ausgrid_filepath, parse_dates=['datetime'])

### Training-Testing Dataset Creation

We will create two testing datasets. The first will evaluate model performance on unseen customers, using models trained to predict PV generation generally rather than for specific customers. It will hold out all three years of data for a subset of customers.

The second will evaluate model performance on unseen days for models trained separately for each customer. Its goal is to predict future generation for known customers. This dataset uses a single chronological split, so it may not include data from all seasons.

In [3]:
unseen_customer_testing_split = 0.15
unseen_datetime_testing_split = 0.15

First, we merse the smart-meter data and weather data on `datetime`.

In [42]:
data = ausgrid_df.merge(open_meteo_df, on='datetime', how='left').sort_values('datetime')

# Check each customer's timestamps against the full expected sequence
expected_timestamps = pd.date_range(
    start=data["datetime"].min(),
    end=data["datetime"].max(),
    freq="30min"
)

customers_with_missing_rows = []

for customer, customer_data in data.groupby("Customer", sort=False):
    customer_timestamps = pd.DatetimeIndex(customer_data["datetime"].unique())
    missing = expected_timestamps.difference(customer_timestamps)
    if not missing.empty: customers_with_missing_rows.append(customer)

# Remove customers with missing timestamps
data = data.loc[~data["Customer"].isin(customers_with_missing_rows)].copy()

print(f"Removed {len(customers_with_missing_rows)} customers")
print(f"Remaining data: {len(data):,} rows, {data['Customer'].nunique()} customers")

Number of Customers with Missing Timestamps: 1
Removed 1 customers
Remaining data: 15,729,792 rows, 299 customers


First, we choose a subset of the customers to hold out. We want to choose a group of customers whose combined data is near the target dataset size specified by `data`. `cumulative_customer_rows` is the running total of rows for shuffled customers. This will allow us to determine how many customers to hold out so their combined rows are closest to the target test-set size.

In [44]:
rng = np.random.default_rng(42)

customer_counts = data.groupby("Customer", sort=False).size()
shuffled_customers = rng.permutation(customer_counts.index.to_numpy())
customer_counts_shuffled = customer_counts.loc[shuffled_customers]
cumulative_customer_rows = customer_counts_shuffled.cumsum()

display(cumulative_customer_rows.rename("Cumulative rows").to_frame().head())

,Cumulative rows
Customer,
185,52608
164,105216
24,157824
294,210432
292,263040


Next, we calculate the desired number of rows in the unseen-customer test set. We find how many shuffled customers give a cumulative row count closest to that target, and convert this into the number of customers to select. Finally, we select that many customers from the start of the shuffled list. Their rows become the test set.

In [45]:
customer_target_rows = unseen_customer_testing_split * len(data)
customer_count_end = (cumulative_customer_rows - customer_target_rows).abs().argmin() + 1
customer_test_customers = shuffled_customers[:customer_count_end]

display(pd.DataFrame({
    "Target rows": [customer_target_rows],
    "Actual rows": [cumulative_customer_rows.iloc[customer_count_end - 1]],
    "Customers selected": [customer_count_end],
}).style.hide(axis='index'))

print(f"Customers selected for test dataset: {sorted(map(int, customer_test_customers))}")

Target rows,Actual rows,Customers selected
2359468.800000,2367360,45


Customers selected for test dataset: [1, 8, 13, 24, 33, 45, 54, 62, 67, 68, 79, 84, 87, 88, 91, 102, 113, 115, 116, 118, 125, 126, 127, 141, 144, 156, 159, 164, 178, 185, 197, 205, 206, 219, 225, 237, 240, 245, 272, 277, 281, 292, 294, 299, 300]


Next we create a Boolean mask `True` for rows whose customer is in the test-customer list, `False` otherwise. We then copy test-customer rows into a separate dataset. We flip the mask in order to select the remaining rows. 

In [46]:
customer_test_mask = data['Customer'].isin(customer_test_customers)
customer_test_data = data.loc[customer_test_mask].copy()
remaining_data = data.loc[~customer_test_mask]

print(f"Customer test data: {len(customer_test_data):,} rows, {customer_test_data['Customer'].nunique()} customers")
print(f"Remaining data: {len(remaining_data):,} rows, {remaining_data['Customer'].nunique()} customers")

Customer test data: 2,367,360 rows, 45 customers
Remaining data: 13,362,432 rows, 254 customers


Next, we create the datetime dataset.
The cumulative rows shows the total number of rows that would be in the testing dataset if the data was split at each timestamp.
this allows us to see where to split the dataset to get the testing dataset to have closest to the specified number of rows.
we calculate the number of rows we want for the test dataset, then chose the timestamp that will give closest to this number of rows.

In [47]:
rows_per_datetime = remaining_data.groupby('datetime', sort=True).size()
display(rows_per_datetime.rename('Rows Per Datetime').to_frame().head())

cumulative_tail_rows = rows_per_datetime.iloc[::-1].cumsum()
display(cumulative_tail_rows.rename('Cumulative Tail Rows').to_frame().head())

datetime_target_rows = (unseen_datetime_testing_split / (1 - unseen_customer_testing_split)) * len(remaining_data)
datetime_start = (cumulative_tail_rows.sub(datetime_target_rows).abs().idxmin())
print(f'Test Dataset Start Datetime: {datetime_start}')

,Rows Per Datetime
datetime,
2010-07-01 00:00:00,254
2010-07-01 00:30:00,254
2010-07-01 01:00:00,254
2010-07-01 01:30:00,254
2010-07-01 02:00:00,254


,Cumulative Tail Rows
datetime,
2013-06-30 23:30:00,254
2013-06-30 23:00:00,508
2013-06-30 22:30:00,762
2013-06-30 22:00:00,1016
2013-06-30 21:30:00,1270


Test Dataset Start Datetime: 2012-12-19 14:00:00


Next, we select rows on or after the start timestamp. Early rows become the training dataset. Because the training dataset now has fewer unique datetimes than the unseen customer testing set, we further trim the unseen customer testing set. Finally, we take the only the data with both unseen customers and unseen timestamps as our third testing dataset.

In [50]:
datetime_test_mask = remaining_data['datetime'] >= datetime_start
datetime_test_data = remaining_data.loc[datetime_test_mask].copy()
train_data = remaining_data.loc[~datetime_test_mask].copy()

# Keep unseen-customer test rows only for timestamps also present in the training set
train_timestamps = train_data['datetime'].unique()
customer_test_data = customer_test_data.loc[
    customer_test_data['datetime'].isin(train_timestamps)].copy()

# Hold out rows for unseen customers at timestamps excluded from training,
customer_datetime_test_mask = (
    data['Customer'].isin(customer_test_customers)
    & ~data['datetime'].isin(train_timestamps)
)
unseen_customer_datetime_test_data = data.loc[customer_datetime_test_mask].copy()

display(pd.DataFrame({
    'Dataset': [
        'Train',
        'Unseen datetime test',
        'Unseen customer test',
        'Unseen customer + datetime test'
    ],
    'Rows': [
        len(train_data),
        len(datetime_test_data),
        len(customer_test_data),
        len(unseen_customer_datetime_test_data)
    ],
    'Customers': [
        train_data['Customer'].nunique(),
        datetime_test_data['Customer'].nunique(),
        customer_test_data['Customer'].nunique(),
        unseen_customer_datetime_test_data['Customer'].nunique()
    ],
    'Unique timestamps': [
        train_data['datetime'].nunique(),
        datetime_test_data['datetime'].nunique(),
        customer_test_data['datetime'].nunique(),
        unseen_customer_datetime_test_data['datetime'].nunique()
    ]
}).style.hide(axis='index'))

Dataset,Rows,Customers,Unique timestamps
Train,11004296,254,43324
Unseen datetime test,2358136,254,9284
Unseen customer test,1949580,45,43324
Unseen customer + datetime test,417780,45,9284


## Average Curve Baseline

PV generation follows two main cyclical patterns: daily and seasonal. The daily curve is generation is zero until sunrise, then rises until midday, then decreases gain until it is zero again at sunset. the seasonal patter is that generation is generally greater in the summer, less during fall and spring, and the least during winter. This regression model predicts the PV generation (`Pgen`) from the historical average at the time time-of-year (month) and time-of-day. This essentially means that every day of each month gets averaged together to create the average pv curve.

we train this model for two scenarios. in the first scenario, we train on historical pv generation for a set of customers, in order to predict the future pv generation for those same customers for unseen timestamps. In the second scenario, we have the same training dataset, but want to test for unseen customers on the same datetimes.